In [ ]:
# /// script
# requires-python = ">=3.12"
# dependencies = []
# ///

In [7]:
import re
import pathlib
import glob

import pandas as pd

In [2]:
Data_Root_Dir = './data'
data_path = pathlib.Path(Data_Root_Dir)

In [ ]:
config = '''
{
    "type": "dir",
    "name": "date",
    "discription": "数据日期",
    "parser": {
        "type": "time",
        "format": "%Y%m%d",
    },
    "child": {
        "type": "dir",
        "name": "module",
        "discription": "采集点位",
        "parser": {
            "type": "time",
            "format": "%H%M%S",
        },
        "child": {
            "type": "dir",
            "discription": "实验名称",
            "parser": {
                "type": "str",
            },
        }
    }
}'''

In [3]:
# 扫描data文件夹下的日期

date_dirs = list(data_path.glob('20'+'[0-9]'*6))
date_df = pd.DataFrame(date_dirs, columns=['date_dir'])
date_df['date'] = date_df['date_dir'].apply(lambda x: x.name)
date_df['date'] = pd.to_datetime(date_df['date'])
date_df.set_index('date', inplace=True)
date_df

,date_dir
date,
2025-06-11,data/20250611


In [4]:
date_df['tasks_dir'] = date_df['date_dir'].apply(lambda x: list(x.glob('20'+'[0-9]'*9)))
date_df = date_df.explode('tasks_dir')
date_df['tasks'] = date_df['tasks_dir'].apply(lambda x: x.name)
date_df['task_id'] = date_df['tasks'].apply(lambda x: x[8:])

date_df

,date_dir,tasks_dir,tasks,task_id
date,,,,
2025-06-11,data/20250611,data/20250611/20250611004,20250611004,004
2025-06-11,data/20250611,data/20250611/20250611003,20250611003,003
2025-06-11,data/20250611,data/20250611/20250611002,20250611002,002
2025-06-11,data/20250611,data/20250611/20250611001,20250611001,001


In [5]:
experiments_df = date_df.copy()
experiments_df['experiments_target_dir'] = experiments_df['tasks_dir'].apply(lambda x: list(x.glob('target/*')))
experiments_df = experiments_df[experiments_df['experiments_target_dir'].apply(len) > 0]
experiments_df = experiments_df.explode('experiments_target_dir')
experiments_df['experiment_name'] = experiments_df['experiments_target_dir'].apply(lambda x: str(x.name))


experiments_df

,date_dir,tasks_dir,tasks,task_id,experiments_target_dir,experiment_name
date,,,,,,
2025-06-11,data/20250611,data/20250611/20250611004,20250611004,004,data/20250611/20250611004/target/2-平顶光线偏振100%-15秒,2-平顶光线偏振100%-15秒
2025-06-11,data/20250611,data/20250611/20250611004,20250611004,004,data/20250611/20250611004/target/1,1
2025-06-11,data/20250611,data/20250611/20250611004,20250611004,004,data/20250611/20250611004/target/4-平顶光线偏振100%-15s,4-平顶光线偏振100%-15s
2025-06-11,data/20250611,data/20250611/20250611004,20250611004,004,data/20250611/20250611004/target/3,3
2025-06-11,data/20250611,data/20250611/20250611002,20250611002,002,data/20250611/20250611002/target/1,1
2025-06-11,data/20250611,data/20250611/20250611002,20250611002,002,data/20250611/20250611002/target/0,0
2025-06-11,data/20250611,data/20250611/20250611002,20250611002,002,data/20250611/20250611002/target/4,4
2025-06-11,data/20250611,data/20250611/20250611002,20250611002,002,data/20250611/20250611002/target/5,5
2025-06-11,data/20250611,data/20250611/20250611002,20250611002,002,data/20250611/20250611002/target/6,6


In [8]:
near_spots_df = date_df.copy()
near_spots_df['experiments_near_spot_dir'] = near_spots_df['date_dir'].apply(lambda x: list(x.glob('**/digitaloptical4Floor/光瞳image/*')))

near_spots_df = near_spots_df.explode('experiments_near_spot_dir')
near_spots_df['dir_name'] = near_spots_df['experiments_near_spot_dir'].apply(lambda x: str(x.name))

def parse_string(input_str):
    # 匹配时间部分，格式为 YYYYMMDD HH：MM：SS
    time_pattern = re.compile(r'(\d{8} \d{2}：\d{2}：\d{2})')
    # 匹配括号内的名称部分
    name_pattern = re.compile(r'\((.*?)\)')

    time_match = time_pattern.search(input_str)
    name_match = name_pattern.search(input_str)

    time_part = time_match.group(1) if time_match else None
    name_part = name_match.group(1) if name_match else None

    return time_part, name_part


# near_spots_df['tmp'] = near_spots_df['dir_name'].apply()


near_spots_df = near_spots_df.join(near_spots_df['dir_name'].apply(parse_string).apply(pd.Series))
near_spots_df.columns = list(near_spots_df.columns)[:-2]+['record_time', 'name']
near_spots_df['record_time'] = near_spots_df['record_time'].apply(lambda x: x.replace('：', ':'))
near_spots_df['record_time'] = pd.to_datetime(near_spots_df['record_time'], format='%Y%m%d %H:%M:%S')

near_spots_df

,date_dir,tasks_dir,tasks,task_id,experiments_near_spot_dir,dir_name,record_time,name
date,,,,,,,,
2025-06-11,data/20250611,data/20250611/20250611004,20250611004,004,data/20250611/20250611004/digitaloptical4Floor...,20250611 16：57：52(3-100%平顶31),2025-06-11 16:57:52,3-100%平顶31
2025-06-11,data/20250611,data/20250611/20250611004,20250611004,004,data/20250611/20250611004/digitaloptical4Floor...,20250611 16：57：52(3-100%平顶31),2025-06-11 16:34:59,1-100%平顶18
2025-06-11,data/20250611,data/20250611/20250611004,20250611004,004,data/20250611/20250611004/digitaloptical4Floor...,20250611 16：57：52(3-100%平顶31),2025-06-11 16:47:02,2-100%平顶24
2025-06-11,data/20250611,data/20250611/20250611004,20250611004,004,data/20250611/20250611004/digitaloptical4Floor...,20250611 16：57：52(3-100%平顶31),2025-06-11 16:29:33,18束弱光平顶
2025-06-11,data/20250611,data/20250611/20250611004,20250611004,004,data/20250611/20250611004/digitaloptical4Floor...,20250611 16：57：52(3-100%平顶31),2025-06-11 15:36:59,1-20%平顶31
...,...,...,...,...,...,...,...,...
2025-06-11,data/20250611,data/20250611/20250611001,20250611001,001,data/20250611/20250611001/digitaloptical4Floor...,20250611 15：08：37(3-100%平顶33),2025-06-11 16:19:22,8-100%平顶18
2025-06-11,data/20250611,data/20250611/20250611001,20250611001,001,data/20250611/20250611001/digitaloptical4Floor...,20250611 15：08：37(3-100%平顶33),2025-06-11 15:04:54,2-100%平顶31
2025-06-11,data/20250611,data/20250611/20250611001,20250611001,001,data/20250611/20250611001/digitaloptical4Floor...,20250611 15：08：37(3-100%平顶33),2025-06-11 14:50:05,全子束弱光平顶


In [ ]:
# scan .tiff files near_spots_df's experiments_near_spot_dir, and save the paths to near_spots_df
def scan_tiff_files(experiments_near_spot_dir):
    tiff_files = experiments_near_spot_dir.glob('*.tiff')
    # read tiff image, filter out the max value less than 100
    tiff_files = [tiff_file for tiff_file in tiff_files if tiffio.imread(tiff_file).max() > 100]
    return tiff_files   


